# Удаление фона в режиме реального времени

В этом ноутбуке реализовано быстрое/малозагруженное для процессора удаление фона в режиме с использованием
`MediaPipe Selfie Segmentation` and `OpenCV`.

Реализованные фичи:
- ввод веб-камеры или видеофайла,
- три режима визуализации: `blur`, `color`, `image`,
- обработанный тест FPS (только сегментация + композиция),

In [2]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from pathlib import Path
import platform
import statistics
import subprocess
import time

import cv2
import mediapipe as mp
import numpy as np


cwd = Path.cwd().resolve()
PROJECT_DIR = cwd
ASSETS_DIR = PROJECT_DIR / "assets"
TMP_DIR = PROJECT_DIR / "tmp"
DEMO_DIR = PROJECT_DIR / "demo"
DEFAULT_BACKGROUND_PATH = ASSETS_DIR / "abstract_studio_background.png"

TMP_DIR.mkdir(parents=True, exist_ok=True)
DEMO_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")
print(f"Background image: {DEFAULT_BACKGROUND_PATH}")

Project dir: /Users/azamath/Desktop/vk_maga/deep_cv/deep-cv-hw2
Background image: /Users/azamath/Desktop/vk_maga/deep_cv/deep-cv-hw2/assets/abstract_studio_background.png


In [3]:
def ensure_odd(value: int) -> int:
    value = max(3, int(value))
    return value if value % 2 == 1 else value + 1


def normalize_source(source):
    if isinstance(source, str) and source.isdigit():
        return int(source)
    return source


def resize_frame(frame: np.ndarray, width: int | None, height: int | None) -> np.ndarray:
    if not width or not height:
        return frame
    if frame.shape[1] == width and frame.shape[0] == height:
        return frame
    return cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)


def resize_for_inference(frame: np.ndarray, inference_width: int) -> np.ndarray:
    if inference_width <= 0 or frame.shape[1] <= inference_width:
        return frame
    scale = inference_width / frame.shape[1]
    inference_height = max(2, int(round(frame.shape[0] * scale)))
    return cv2.resize(frame, (inference_width, inference_height), interpolation=cv2.INTER_AREA)


def soft_alpha(mask: np.ndarray, threshold: float) -> np.ndarray:
    alpha = (mask - threshold) / max(1e-6, 1.0 - threshold)
    alpha = np.clip(alpha, 0.0, 1.0)
    alpha = cv2.GaussianBlur(alpha, (11, 11), 0)
    return np.clip(alpha, 0.0, 1.0)


def draw_overlay(frame: np.ndarray, processed_fps: float, mode: str, frames_processed: int) -> np.ndarray:
    lines = [
        f"Mode: {mode}",
        f"Processed FPS: {processed_fps:.2f}",
        f"Frames: {frames_processed}",
    ]
    y = 30
    for line in lines:
        cv2.putText(
            frame,
            line,
            (16, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 0, 0),
            3,
            cv2.LINE_AA,
        )
        cv2.putText(
            frame,
            line,
            (16, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (245, 245, 245),
            1,
            cv2.LINE_AA,
        )
        y += 28
    return frame


def get_cpu_name() -> str:
    if platform.system() == "Darwin":
        try:
            return subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"],
                text=True,
            ).strip()
        except Exception:
            pass
    return platform.processor() or platform.machine()


@dataclass
class PipelineConfig:
    model_selection: int = 1
    background_mode: str = "image"
    background_color: tuple[int, int, int] = (32, 180, 240)
    blur_kernel: int = 55
    mask_threshold: float = 0.15
    temporal_smoothing: float = 0.70
    infer_every_n: int = 1
    inference_width: int = 256
    target_width: int = 640
    target_height: int = 480
    flip_camera: bool = True
    background_image_path: Path | None = DEFAULT_BACKGROUND_PATH

In [4]:
class BackgroundRemover:
    def __init__(self, config: PipelineConfig):
        self.config = config
        self.segmenter = mp.solutions.selfie_segmentation.SelfieSegmentation(
            model_selection=config.model_selection
        )
        self.previous_mask: np.ndarray | None = None
        self.cached_mask: np.ndarray | None = None
        self.frame_index = 0
        self.cached_background: np.ndarray | None = None
        self.cached_background_shape: tuple[int, int] | None = None

    def _infer_mask(self, frame: np.ndarray) -> np.ndarray:
        inference_frame = resize_for_inference(frame, self.config.inference_width)
        rgb = cv2.cvtColor(inference_frame, cv2.COLOR_BGR2RGB)
        result = self.segmenter.process(rgb)
        if result.segmentation_mask is None:
            raise RuntimeError("Segmentation mask was not produced.")
        mask = result.segmentation_mask.astype(np.float32)
        return cv2.resize(mask, (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_LINEAR)

    def _get_background(self, frame: np.ndarray) -> np.ndarray:
        mode = self.config.background_mode
        if mode == "blur":
            kernel = ensure_odd(self.config.blur_kernel)
            return cv2.GaussianBlur(frame, (kernel, kernel), 0)
        if mode == "color":
            background = np.empty_like(frame)
            background[:] = np.array(self.config.background_color, dtype=np.uint8)
            return background
        if mode == "image":
            path = Path(self.config.background_image_path or DEFAULT_BACKGROUND_PATH)
            if not path.exists():
                raise FileNotFoundError(f"Background image not found: {path}")
            if self.cached_background is None or self.cached_background_shape != frame.shape[:2]:
                image = cv2.imread(str(path))
                if image is None:
                    raise RuntimeError(f"Failed to read background image: {path}")
                self.cached_background = cv2.resize(
                    image,
                    (frame.shape[1], frame.shape[0]),
                    interpolation=cv2.INTER_LINEAR,
                )
                self.cached_background_shape = frame.shape[:2]
            return self.cached_background.copy()
        raise ValueError(f"Unsupported background mode: {mode}")

    def process(self, frame: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        self.frame_index += 1
        infer_every_n = max(1, self.config.infer_every_n)
        should_infer = self.cached_mask is None or (self.frame_index - 1) % infer_every_n == 0

        if should_infer:
            raw_mask = self._infer_mask(frame)
            if self.previous_mask is None:
                smoothed_mask = raw_mask
            else:
                factor = float(np.clip(self.config.temporal_smoothing, 0.0, 0.98))
                smoothed_mask = factor * self.previous_mask + (1.0 - factor) * raw_mask
            self.previous_mask = smoothed_mask
            self.cached_mask = smoothed_mask

        alpha = soft_alpha(self.cached_mask, self.config.mask_threshold)
        background = self._get_background(frame)

        composite = (
            frame.astype(np.float32) * alpha[..., None]
            + background.astype(np.float32) * (1.0 - alpha[..., None])
        )
        return composite.astype(np.uint8), alpha

    def close(self) -> None:
        self.segmenter.close()

In [5]:
def run_realtime_demo(
    source=0,
    *,
    background_mode: str = "image",
    background_image_path: str | Path | None = DEFAULT_BACKGROUND_PATH,
    background_color: tuple[int, int, int] = (32, 180, 240),
    width: int = 640,
    height: int = 480,
    inference_width: int = 256,
    infer_every_n: int = 1,
    flip_camera: bool = True,
    show_window: bool = True,
    output_path: str | Path | None = None,
    max_frames: int | None = None,
) -> dict:
    source = normalize_source(source)
    capture = cv2.VideoCapture(source)
    if not capture.isOpened():
        raise RuntimeError(f"Failed to open source: {source}")

    if isinstance(source, int):
        capture.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        capture.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

    config = PipelineConfig(
        background_mode=background_mode,
        background_color=background_color,
        inference_width=inference_width,
        infer_every_n=infer_every_n,
        target_width=width,
        target_height=height,
        flip_camera=flip_camera,
        background_image_path=Path(background_image_path) if background_image_path else None,
    )
    remover = BackgroundRemover(config)

    writer = None
    display_fps_window = deque(maxlen=30)
    process_times: list[float] = []
    frames_processed = 0
    source_fps = capture.get(cv2.CAP_PROP_FPS)
    source_fps = source_fps if source_fps and source_fps > 1 else 20.0
    resolution = "unknown"

    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break

            frame = resize_frame(frame, width, height)
            if isinstance(source, int) and flip_camera:
                frame = cv2.flip(frame, 1)
            resolution = f"{frame.shape[1]}x{frame.shape[0]}"

            t0 = time.perf_counter()
            composite, _ = remover.process(frame)
            elapsed = time.perf_counter() - t0

            process_times.append(elapsed)
            display_fps_window.append(elapsed)
            frames_processed += 1

            processed_fps = len(display_fps_window) / max(sum(display_fps_window), 1e-6)
            output_frame = draw_overlay(
                composite.copy(),
                processed_fps=processed_fps,
                mode=background_mode,
                frames_processed=frames_processed,
            )

            if output_path and writer is None:
                output_path = Path(output_path)
                output_path.parent.mkdir(parents=True, exist_ok=True)
                writer = cv2.VideoWriter(
                    str(output_path),
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    source_fps,
                    (output_frame.shape[1], output_frame.shape[0]),
                )
            if writer is not None:
                writer.write(output_frame)

            if show_window:
                cv2.imshow("Background Removal (press q to quit)", output_frame)
                key = cv2.waitKey(1) & 0xFF
                if key in (ord("q"), 27):
                    break

            if max_frames is not None and frames_processed >= max_frames:
                break
    finally:
        capture.release()
        if writer is not None:
            writer.release()
        remover.close()
        if show_window:
            cv2.destroyAllWindows()

    total_process_time = sum(process_times)
    return {
        "frames_processed": frames_processed,
        "resolution": resolution,
        "avg_processed_fps": frames_processed / max(total_process_time, 1e-6),
        "avg_latency_ms": 1000.0 * statistics.mean(process_times) if process_times else float("nan"),
        "background_mode": background_mode,
        "infer_every_n": infer_every_n,
        "inference_width": inference_width,
        "output_path": str(output_path) if output_path else None,
    }


def benchmark_processed_fps(
    source,
    *,
    background_mode: str = "image",
    background_image_path: str | Path | None = DEFAULT_BACKGROUND_PATH,
    background_color: tuple[int, int, int] = (32, 180, 240),
    width: int = 640,
    height: int = 480,
    inference_width: int = 256,
    infer_every_n: int = 1,
    flip_camera: bool = False,
    max_frames: int = 180,
    warmup_frames: int = 15,
) -> dict:
    source = normalize_source(source)
    capture = cv2.VideoCapture(source)
    if not capture.isOpened():
        raise RuntimeError(f"Failed to open source: {source}")

    if isinstance(source, int):
        capture.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        capture.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

    config = PipelineConfig(
        background_mode=background_mode,
        background_color=background_color,
        inference_width=inference_width,
        infer_every_n=infer_every_n,
        target_width=width,
        target_height=height,
        flip_camera=flip_camera,
        background_image_path=Path(background_image_path) if background_image_path else None,
    )
    remover = BackgroundRemover(config)

    process_times: list[float] = []
    frames_seen = 0
    resolution = "unknown"

    try:
        while len(process_times) < max_frames:
            ok, frame = capture.read()
            if not ok:
                break
            frame = resize_frame(frame, width, height)
            if isinstance(source, int) and flip_camera:
                frame = cv2.flip(frame, 1)
            resolution = f"{frame.shape[1]}x{frame.shape[0]}"

            t0 = time.perf_counter()
            _ = remover.process(frame)
            elapsed = time.perf_counter() - t0

            frames_seen += 1
            if frames_seen > warmup_frames:
                process_times.append(elapsed)
    finally:
        capture.release()
        remover.close()

    total_process_time = sum(process_times)
    return {
        "frames_benchmarked": len(process_times),
        "warmup_frames": warmup_frames,
        "resolution": resolution,
        "avg_processed_fps": len(process_times) / max(total_process_time, 1e-6),
        "avg_latency_ms": 1000.0 * statistics.mean(process_times) if process_times else float("nan"),
        "background_mode": background_mode,
        "infer_every_n": infer_every_n,
        "inference_width": inference_width,
        "cpu": get_cpu_name(),
        "platform": platform.platform(),
    }

In [6]:
sample_video = TMP_DIR / "sample_person.mp4"
demo_output = DEMO_DIR / "background_removal_demo.mp4"

system_info = {
    "cpu": get_cpu_name(),
    "platform": platform.platform(),
    "python": platform.python_version(),
    "opencv": cv2.__version__,
    "mediapipe": mp.__version__,
}
system_info

{'cpu': 'Apple M3',
 'platform': 'macOS-15.3-x86_64-i386-64bit',
 'python': '3.10.17',
 'opencv': '4.11.0',
 'mediapipe': '0.10.14'}

In [6]:
if sample_video.exists():
    benchmark_result = benchmark_processed_fps(
        sample_video,
        background_mode="image",
        background_image_path=DEFAULT_BACKGROUND_PATH,
        width=640,
        height=480,
        inference_width=256,
        infer_every_n=1,
        flip_camera=False,
        max_frames=180,
        warmup_frames=15,
    )
    print(benchmark_result)
else:
    print(f"Sample video not found: {sample_video}")

I0000 00:00:1776104780.186603  307751 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776104780.192166  307855 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


{'frames_benchmarked': 180, 'warmup_frames': 15, 'resolution': '640x480', 'avg_processed_fps': 100.51120860575563, 'avg_latency_ms': 9.949139144494742, 'background_mode': 'image', 'infer_every_n': 1, 'inference_width': 256, 'cpu': 'Apple M3', 'platform': 'macOS-15.3-x86_64-i386-64bit'}


In [7]:
if sample_video.exists():
    demo_result = run_realtime_demo(
        sample_video,
        background_mode="image",
        background_image_path=DEFAULT_BACKGROUND_PATH,
        width=640,
        height=480,
        inference_width=256,
        infer_every_n=1,
        flip_camera=False,
        show_window=False,
        max_frames=180,
        output_path=demo_output,
    )
    print(demo_result)
else:
    print(f"Sample video not found: {sample_video}")

I0000 00:00:1776104782.918280  307751 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M3
W0000 00:00:1776104782.919762  307936 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


{'frames_processed': 180, 'resolution': '640x480', 'avg_processed_fps': 97.2427164840962, 'avg_latency_ms': 10.28354653341618, 'background_mode': 'image', 'infer_every_n': 1, 'inference_width': 256, 'output_path': '/Users/azamath/Desktop/vk_maga/deep_cv/hw2/demo/background_removal_demo.mp4'}


## Пряма трансляция с веб-камеры

Нажмите "q", чтобы остановить.

In [ ]:
run_realtime_demo(
    source=0,
    background_mode="image",
    background_image_path=DEFAULT_BACKGROUND_PATH,
    width=640,
    height=480,
    inference_width=256,
    infer_every_n=1,
    flip_camera=True,
    show_window=True,
)

I0000 00:00:1776107093.011752  352017 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776107093.028944  352442 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2026-04-13 22:04:53.175 Python[44801:352017] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/6g/_8jwr6410mj3v82r2qt782v40000gn/T/org.python.python.savedState
2026-04-13 22:04:53.634 Python[44801:352017] +[IMKClient subclass]: chose IMKClient_Modern
2026-04-13 22:04:53.634 Python[44801:352017] +[IMKInputSession subclass]: chose IMKInputSession_Modern


{'frames_processed': 59,
 'resolution': '640x480',
 'avg_processed_fps': 45.058342347748564,
 'avg_latency_ms': 22.19344849134174,
 'background_mode': 'image',
 'infer_every_n': 1,
 'inference_width': 256,
 'output_path': None}

: 